# 👄 EidCloud Lip-Sync Analyzer Quickstart

Welcome to **eidcloud-lipsync-analyzer**! This interactive notebook demonstrates how to measure audio-to-video drift (AV sync offset in milliseconds), calculate ITU-R compliant sync scores, and generate timeline visual graphs using pure PHP 8.2+ and FFmpeg.

### Step 1: Environment Setup
Install PHP and FFmpeg in your Google Colab instance, clone the repository, and make the CLI executable.

In [ ]:
!apt-get update -qq && apt-get install -y -qq php-cli ffmpeg git
!if [ ! -d "eidcloud-lipsync-analyzer" ]; then git clone https://github.com/eidcloud/eidcloud-lipsync-analyzer.git; fi
%cd eidcloud-lipsync-analyzer
!chmod +x bin/eidcloud-lipsync
!php bin/eidcloud-lipsync --help

In [ ]:
# Generate a synchronized test video and an artificially delayed (lagging audio) video
!ffmpeg -y -f lavfi -i "sine=frequency=880:duration=3" -f lavfi -i "color=c=navy:s=320x240:d=3" -c:v libx264 -pix_fmt yuv420p -c:a aac sync_test.mp4
# Create an out-of-sync video with audio delayed by 150ms
!ffmpeg -y -i sync_test.mp4 -itsoffset 0.150 -i sync_test.mp4 -map 0:v:0 -map 1:a:0 -c copy desync_test.mp4

print("=== 1. Analyzing Synchronized Clip ===")
!php bin/eidcloud-lipsync analyze sync_test.mp4 --json

print("\n=== 2. Analyzing Desynchronized Clip (+150ms delay) ===")
!php bin/eidcloud-lipsync analyze desync_test.mp4 --json

print("\n=== 3. Rendering SVG Timeline ===")
!php bin/eidcloud-lipsync timeline sync_test.mp4 --out=sync_timeline.svg

from IPython.display import SVG, display
display(SVG('sync_timeline.svg'))